In [1]:
import numpy as np

from scipy.spatial.transform import Rotation
from scipy.optimize import minimize

from pyscf import scf, qmmm, gto

import contextlib
import io

In [2]:
# ============================================================
# Constants
# ============================================================

HARTREE_TO_KJMOL = 2625.49962

In [3]:
# ============================================================
# Lennard-Jones energy
# ============================================================

def lj_energy(
    qm_coords,
    qm_atom_types,
    mm_coords,
    mm_atom_types,
    lj_params
):
    """
    Calculate QM-MM Lennard-Jones energy.

    Parameters
    ----------
    qm_coords : (N,3) array
        QM coordinates in Angstrom.

    qm_atom_types : list
        LJ atom type for each QM atom.

    mm_coords : (M,3) array
        MM LJ-site coordinates in Angstrom.

    mm_atom_types : list
        LJ atom type for each MM particle.

    lj_params : dict
        Dictionary containing sigma (Angstrom) and
        epsilon (kJ/mol) for each atom type.

    Returns
    -------
    total : float
        Total QM-MM LJ energy in kJ/mol.
    """

    total = 0.0

    # print()
    # print("Individual QM-MM LJ interactions")
    # print("----------------------------------------")

    for i, r_qm in enumerate(qm_coords):

        for j, r_mm in enumerate(mm_coords):

            qm_type = qm_atom_types[i]
            mm_type = mm_atom_types[j]

            r = np.linalg.norm(r_qm - r_mm)

            if r == 0.0:
                raise ValueError(
                    f"Zero distance between QM atom {i} "
                    f"and MM atom {j}"
                )

            sigma_qm = lj_params[qm_type]["sigma_A"]
            epsilon_qm = lj_params[qm_type]["epsilon_kJmol"]

            sigma_mm = lj_params[mm_type]["sigma_A"]
            epsilon_mm = lj_params[mm_type]["epsilon_kJmol"]

            # Lorentz-Berthelot mixing rules
            sigma = 0.5 * (sigma_qm + sigma_mm)
            epsilon = np.sqrt(epsilon_qm * epsilon_mm)

            sr6 = (sigma / r) ** 6
            sr12 = sr6 ** 2

            e = 4.0 * epsilon * (sr12 - sr6)

            total += e

            # print(
            #     f"QM {i:2d} ({qm_type:6s})  "
            #     f"MM {j:2d} ({mm_type:6s})  "
            #     f"r = {r:8.4f} Å   "
            #     f"E = {e:14.6f} kJ/mol"
            # )

    return total

def water_from_variables(x):
    """
    Convert 6 optimization variables into TIP4P-D coordinates.

    x[0:3] = oxygen position in Angstrom
    x[3:6] = rotation vector in radians
    """

    # Oxygen position
    O = np.array(x[:3])

    # Rotation vector
    rotvec = np.array(x[3:6])

    R = Rotation.from_rotvec(rotvec)

    # Rotate the reference geometry around O
    H1 = O + R.apply(water_H1_ref)
    H2 = O + R.apply(water_H2_ref)
    M  = O + R.apply(water_M_ref)

    return O, H1, H2, M

# ============================================================
# Total QM/MM + LJ interaction energy
# ============================================================

def total_qm_water_energy(x, verbose=False):

    # --------------------------------------------------------
    # Build TIP4P-D water
    # --------------------------------------------------------

    water_O, water_H1, water_H2, water_M = \
        water_from_variables(x)

    # --------------------------------------------------------
    # TIP4P-D electrostatic sites
    # --------------------------------------------------------

    mm_charge_coords = np.array([
        water_H1,
        water_H2,
        water_M
    ])

    mm_charges = np.array([
        +0.58,
        +0.58,
        -1.16
    ])

    # --------------------------------------------------------
    # PySCF QM/MM electrostatics
    # --------------------------------------------------------

    mf_qm = scf.RHF(mol)
    mf_qm.verbose = 0
    
    mf_qmmm = qmmm.mm_charge(
        mf_qm,
        mm_charge_coords,
        mm_charges,
        unit='Angstrom'
    )

    mf_qmmm.verbose = 0
    energy_qmmm_hartree = mf_qmmm.kernel()

    # with contextlib.redirect_stdout(io.StringIO()):
    #     energy_qmmm_hartree = mf_qmmm.kernel()
    
    energy_qmmm_kjmol = (
        energy_qmmm_hartree * HARTREE_TO_KJMOL
    )

    # --------------------------------------------------------
    # QM/MM electrostatic interaction
    # --------------------------------------------------------

    energy_electrostatic = (
        energy_qmmm_kjmol
        - energy_qm_kjmol
    )

    # --------------------------------------------------------
    # LJ
    #
    # Only TIP4P oxygen is an LJ site.
    # --------------------------------------------------------

    mm_lj_coords = np.array([
        water_O
    ])

    mm_lj_types = [
        "TIP4P_O"
    ]

    energy_lj = lj_energy(
        qm_coords,
        qm_atom_types,
        mm_lj_coords,
        mm_lj_types,
        lj_params
    )

    # --------------------------------------------------------
    # Total
    # --------------------------------------------------------

    energy_total = (
        energy_electrostatic
        + energy_lj
    )

    if verbose:

        print()
        print("--------------------------------------------")
        print("QM/MM water energy")
        print("--------------------------------------------")
        print(f"Electrostatic = {energy_electrostatic:12.6f} kJ/mol")
        print(f"LJ            = {energy_lj:12.6f} kJ/mol")
        print(f"Total         = {energy_total:12.6f} kJ/mol")
        print("--------------------------------------------")

    return energy_total

def total_qm_water_energy_xyz(xyz):

    # --------------------------------------------------------
    # Build rigid water at xyz
    # --------------------------------------------------------

    water_O, water_H1, water_H2, water_M = \
        water_from_xyz(xyz)

    # --------------------------------------------------------
    # TIP4P-D electrostatic sites
    # --------------------------------------------------------

    mm_charge_coords = np.array([
        water_H1,
        water_H2,
        water_M
    ])

    mm_charges = np.array([
        +0.58,
        +0.58,
        -1.16
    ])

    # --------------------------------------------------------
    # QM/MM electrostatic energy
    # --------------------------------------------------------

    mf_qm = scf.RHF(mol)
    mf_qm.verbose = 0

    mf_qmmm = qmmm.mm_charge(
        mf_qm,
        mm_charge_coords,
        mm_charges,
        unit='Angstrom'
    )

    mf_qmmm.verbose = 0

    energy_qmmm_hartree = mf_qmmm.kernel()

    energy_qmmm_kjmol = (
        energy_qmmm_hartree * HARTREE_TO_KJMOL
    )

    # --------------------------------------------------------
    # Electrostatic interaction
    # --------------------------------------------------------

    energy_electrostatic = (
        energy_qmmm_kjmol
        - energy_qm_kjmol
    )

    # --------------------------------------------------------
    # LJ interaction
    # --------------------------------------------------------

    energy_lj = lj_energy(
        qm_coords,
        qm_atom_types,
        np.array([water_O]),
        ["TIP4P_O"],
        lj_params
    )

    # --------------------------------------------------------
    # Total
    # --------------------------------------------------------

    energy_total = (
        energy_electrostatic
        + energy_lj
    )

    return energy_total
    
def water_from_xyz(xyz):
    """
    Construct a rigid TIP4P-D water with fixed orientation.

    xyz = [Ox, Oy, Oz] in Angstrom
    """

    O = np.array(xyz)

    H1 = O + np.array([
         0.757,
         0.586,
         0.000
    ])

    H2 = O + np.array([
        -0.757,
         0.586,
         0.000
    ])

    M = O + np.array([
         0.000,
         0.151,
         0.000
    ])

    return O, H1, H2, M

In [4]:
# ============================================================
# TIP4P-D reference geometry
# ============================================================

water_H1_ref = np.array([
     0.757,
     0.586,
     0.000
])

water_H2_ref = np.array([
    -0.757,
     0.586,
     0.000
])

water_M_ref = np.array([
     0.000,
     0.151,
     0.000
])

In [5]:
# ============================================================
# OH-
#
# O-H distance = 0.97 Å
#
# Charge = -1
# ============================================================

mol = gto.M(
    atom='''
        O    0.000    0.000    0.000
        H    0.970    0.000    0.000
    ''',
    basis='sto-3g',
    charge=-1,
    spin=0,
    unit='Angstrom',
    verbose=4
)

System: uname_result(system='Linux', node='Cas2', release='6.1.177-1-MANJARO', version='#1 SMP PREEMPT_DYNAMIC Sat Jul  4 22:18:29 UTC 2026', machine='x86_64')  Threads 32
Python 3.14.6 (main, Jun 15 2026, 11:36:54) [GCC 16.1.1 20260430]
numpy 2.5.1  scipy 1.18.0  h5py 3.16.0
Date: Thu Aug 20 15:39:14 2026
PySCF version 2.14.0
PySCF path  /home/chemistry/venvs/jupyter/lib/python3.14/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 10
[INPUT] charge = -1
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = Angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O      0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0
[INPUT]  2 H      0.970000000000   0.000000000000   0.000000000000 AA    1.833034340828   0.000000000

In [6]:
# ============================================================
# BARE QM ENERGY
# ============================================================

mf_qm = scf.RHF(mol)

energy_qm_hartree = mf_qm.kernel()

energy_qm_kjmol = (
    energy_qm_hartree * HARTREE_TO_KJMOL
)



******** <class 'pyscf.scf.hf.RHF'> ********
method = RHF
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = /tmp/tmpa05ne1fl
max_memory 4000 MB (current use 178 MB)
Set gradient conv threshold to 3.16228e-05
Initial guess from minao.
init E= -73.9765896919984
  HOMO = -0.336126307462796  LUMO = 0.488509951930397  gap/eV = 22.43950
cycle= 1 E= -73.9300784358367  delta_E= 0.0465  |g|= 0.529  |ddm|=  1.5
  HOMO = 0.414541082916799  LUMO = 1.24739187792405  gap/eV = 22.66302
cycle= 2 E= -74.0560473499752  delta_E= -0.126  |g|= 0.0535  |ddm|= 0.869
  HOMO = 0.251789068399813  LUMO = 1.24576848161728  gap/eV = 27.04756
cycle= 3 E= -74.0573907484134  delta_E= -0.00134  |g|= 0.00458  |ddm|= 0.11
  HOMO = 0.25111268174294  LUMO = 1.24325478899863  gap/eV = 26

In [7]:
# ============================================================
# LJ PARAMETERS
# ============================================================

lj_params = {

    "OH_O": {
        "sigma_A": 3.400,
        "epsilon_kJmol": 0.2508914038369354
    },

    "OH_H": {
        "sigma_A": 1.443,
        "epsilon_kJmol": 0.18390926154006562
    },

    # TEST TIP4P-D oxygen parameters
    "TIP4P_O": {
        "sigma_A": 3.15365,
        "epsilon_kJmol": 0.650194
    }
}


qm_coords = mol.atom_coords(unit='Angstrom')
qm_atom_types = [
    "OH_O",
    "OH_H"
]


x0 = np.array([
    3.5,    # O x (Angstrom)
    0.0,    # O y
    0.0,    # O z

    0.0,    # rotation x (radians)
    0.0,    # rotation y
    0.0     # rotation z
])

print("Initial variables:")
print(x0)

print()
print(
    f"Initial energy = "
    f"{total_qm_water_energy(x0):.6f} kJ/mol"
)

x0_xyz = np.array([
    3.5,
    0.0,
    0.0
])

print(x0_xyz)

Initial variables:
[3.5 0.  0.  0.  0.  0. ]

Initial energy = -16.769262 kJ/mol
[3.5 0.  0. ]


In [10]:
# ============================================================
# Optimize water position + orientation
# ============================================================

# result = minimize(
#     total_qm_water_energy,
#     x0,
#     method="Powell",
#     options={
#         "maxiter": 20,
#         "xtol": 1e-3,
#         "ftol": 1e-3,
#         "disp": True
#     }
# )

result = minimize(
    total_qm_water_energy_xyz,
    x0_xyz,
    method="L-BFGS-B",
    bounds=[
        (1.5, 6.0),    # x
        (-4.0, 4.0),   # y
        (-4.0, 4.0)    # z
    ],
    options={
        "maxiter": 50,
        "ftol": 1e-6,
        "gtol": 1e-4
    }
)
print()
print("======================================================")
print("             WATER OPTIMIZATION")
print("======================================================")

print()
print("Success:")
print(result.success)

print()
print("Message:")
print(result.message)

print()
print("Number of function evaluations:")
print(result.nfev)

# print()
# print("Initial energy:")
# print(
#     f"{total_qm_water_energy(x0):.6f} kJ/mol"
# )

# print()
# print("Final energy:")
# print(
#     f"{result.fun:.6f} kJ/mol"
# )

print()
print("Initial energy:")
print(
    f"{total_qm_water_energy_xyz(x0_xyz):.6f} kJ/mol"
)

print()
print("Final energy:")
print(
    f"{result.fun:.6f} kJ/mol"
)

print()
print("Optimized variables:")
print(result.x)


             WATER OPTIMIZATION

Success:
True

Message:
CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

Number of function evaluations:
48

Initial energy:
-16.769262 kJ/mol

Final energy:
-93.467582 kJ/mol

Optimized variables:
[ 1.60558062e+00 -2.09958871e+00  1.00295144e-04]


In [11]:
# ============================================================
# Energy breakdown at optimized geometry
# ============================================================

O_opt, H1_opt, H2_opt, M_opt = \
    water_from_xyz(result.x)

print("Optimized water:")
print("O :", O_opt)
print("H1:", H1_opt)
print("H2:", H2_opt)
print("M :", M_opt)

# ------------------------------------------------------------
# Electrostatic interaction
# ------------------------------------------------------------

mm_charge_coords = np.array([
    H1_opt,
    H2_opt,
    M_opt
])

mm_charges = np.array([
    +0.58,
    +0.58,
    -1.16
])

mf_qm = scf.RHF(mol)
mf_qm.verbose = 0

mf_qmmm = qmmm.mm_charge(
    mf_qm,
    mm_charge_coords,
    mm_charges,
    unit='Angstrom'
)

mf_qmmm.verbose = 0

energy_qmmm_hartree = mf_qmmm.kernel()

energy_qmmm_kjmol = (
    energy_qmmm_hartree * HARTREE_TO_KJMOL
)

E_elec = (
    energy_qmmm_kjmol
    - energy_qm_kjmol
)

# ------------------------------------------------------------
# LJ interaction
# ------------------------------------------------------------

E_LJ = lj_energy(
    qm_coords,
    qm_atom_types,
    np.array([O_opt]),
    ["TIP4P_O"],
    lj_params
)

# ------------------------------------------------------------
# Total
# ------------------------------------------------------------

E_total = E_elec + E_LJ

print()
print("Energy breakdown:")
print(f"Electrostatic = {E_elec:12.6f} kJ/mol")
print(f"LJ            = {E_LJ:12.6f} kJ/mol")
print(f"Total         = {E_total:12.6f} kJ/mol")

Optimized water:
O : [ 1.60558062e+00 -2.09958871e+00  1.00295144e-04]
H1: [ 2.36258062e+00 -1.51358871e+00  1.00295144e-04]
H2: [ 8.48580616e-01 -1.51358871e+00  1.00295144e-04]
M : [ 1.60558062e+00 -1.94858871e+00  1.00295144e-04]

Energy breakdown:
Electrostatic =  -109.489578 kJ/mol
LJ            =    16.021996 kJ/mol
Total         =   -93.467582 kJ/mol


In [ ]:
O_opt, H1_opt, H2_opt, M_opt = water_from_variables(result.x)

print("O :", O_opt)
print("H1:", H1_opt)
print("H2:", H2_opt)
print("M :", M_opt)

print()

# Electrostatic
mm_charge_coords = np.array([
    H1_opt,
    H2_opt,
    M_opt
])

mm_charges = np.array([
    +0.58,
    +0.58,
    -1.16
])

mf_qmmm = qmmm.mm_charge(
    scf.RHF(mol),
    mm_charge_coords,
    mm_charges,
    unit='Angstrom'
)

mf_qmmm.verbose = 0

energy_qmmm = mf_qmmm.kernel()

energy_qmmm_kjmol = (
    energy_qmmm * HARTREE_TO_KJMOL
)

E_elec = energy_qmmm_kjmol - energy_qm_kjmol

# LJ
E_LJ = lj_energy(
    qm_coords,
    qm_atom_types,
    np.array([O_opt]),
    ["TIP4P_O"],
    lj_params
)

print(f"Electrostatic = {E_elec:12.6f} kJ/mol")
print(f"LJ            = {E_LJ:12.6f} kJ/mol")
print(f"Total         = {E_elec + E_LJ:12.6f} kJ/mol")

In [1]:
ls

 initial_qmmm.xyz                       optimization.xyz
'minimize with H2O rotation.ipynb'      optimized_qmmm.xyz
'minimize with no H2O rotation.ipynb'
